# Floor-plan YOLO training on the GTX 1650

In VS Code, choose **Select Kernel > Python Environments > .venv-training**.
The interpreter is `backend/.venv-training/Scripts/python.exe` (Python 3.12).
Restart the kernel after changing it, then run the cells in order.
First complete `vision/setup_gpu.ps1` from PowerShell to install and verify the environment.
Do not install CPU-only PyTorch into this environment.
The final cell starts 60 epochs of GPU training with batch size 1.


In [1]:
import sys
import torch
import ultralytics

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select backend/.venv-training/Scripts/python.exe "
        "as the notebook kernel, restart it, and rerun this cell."
    )
print("GPU:", torch.cuda.get_device_name(0))
# Execute an operation on the GPU, rather than only checking detection.
x = torch.ones((32, 32), device="cuda:0")
print("CUDA computation:", (x @ x)[0, 0].item())
del x


Python: d:\R26-IT-091\R26-IT-091\backend\.venv-training\Scripts\python.exe
PyTorch: 2.5.1+cu124
Ultralytics: 8.4.48
GPU: NVIDIA GeForce GTX 1650
CUDA computation: 32.0


In [2]:
from pathlib import Path

# Jupyter does not define __file__. Locate backend from the notebook working directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
BASE_DIR = next(
    (p for root in candidates for p in (root, root / "backend", root / "R26-IT-091" / "backend")
     if (p / "datasets" / "blueprints_large" / "data.yaml").is_file()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("Open this notebook from the R26-IT-091 project; blueprints_large/data.yaml was not found.")
DATA_YAML = BASE_DIR / "datasets" / "blueprints_large" / "data.yaml"
print("Backend:", BASE_DIR)
print("Dataset:", DATA_YAML)


Backend: D:\R26-IT-091\R26-IT-091\backend
Dataset: D:\R26-IT-091\R26-IT-091\backend\datasets\blueprints_large\data.yaml


In [3]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
dataset_dir = Path(config.get("path", DATA_YAML.parent))
if not dataset_dir.is_absolute():
    dataset_dir = (DATA_YAML.parent / dataset_dir).resolve()
for split in ("train", "val"):
    image_dir = dataset_dir / config[split]
    label_dir = image_dir.parent / "labels"
    images = [p for p in image_dir.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg")]
    labels = list(label_dir.glob("*.txt"))
    print(f"{split}: {len(images)} images, {len(labels)} labels")
    if not images or not labels:
        raise RuntimeError(f"Images or labels missing for {split}")


train: 800 images, 800 labels
val: 200 images, 200 labels


In [4]:
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this training run.")

# Keep weights and results at consistent locations regardless of Jupyter's working directory.
model = YOLO(str(BASE_DIR / "vision" / "yolov8m.pt"))
results = model.train(
    data=str(DATA_YAML),
    epochs=60,
    imgsz=640,
    batch=1,
    device=0,
    workers=0,
    cache=False,
    amp=True,
    cos_lr=True,
    augment=True,
    patience=15,
    project=str(BASE_DIR / "runs"),
    name="blueprint_model_gpu",
    exist_ok=False,
    save=True,
    plots=True,
)
print("GPU training completed!")
print("Best weights:", model.trainer.best)


New https://pypi.org/project/ultralytics/8.4.175 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.12.6 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\R26-IT-091\R26-IT-091\backend\datasets\blueprints_large\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\R26-IT-091\R26-I

Matplotlib is building the font cache; this may take a moment.


Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

d:\R26-IT-091\R26-IT-091\backend\.venv-training\Lib\site-packages\ultralytics\utils\loss.py:390: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


       1/60      1.57G      3.149      2.644      2.322         94        640: 100% ━━━━━━━━━━━━ 800/800 2.7it/s 4:59<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 2.9it/s 34.6s.3ss
                   all        200      20826      0.107     0.0513     0.0229    0.00756

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/60      1.59G      3.349      2.643      2.188        116        640: 0% ──────────── 0/800  2.0s

...

      53/60      1.59G      2.323       1.99      1.741         46        640: 100% ━━━━━━━━━━━━ 800/800 4.1it/s 3:15<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 7.2it/s 13.9s.2s
                   all        200      20826      0.207     0.0979     0.0571     0.0217

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


d:\R26-IT-091\R26-IT-091\backend\.venv-training\Lib\site-packages\ultralytics\utils\loss.py:390: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


      54/60      1.59G      2.306       1.98      1.737        107        640: 100% ━━━━━━━━━━━━ 800/800 4.2it/s 3:09<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 7.1it/s 14.2s.1s
                   all        200      20826      0.211     0.0991     0.0571      0.022

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      55/60      1.59G      2.646       2.39      2.086         69        640: 0% ──────────── 0/800  0.4s

d:\R26-IT-091\R26-IT-091\backend\.venv-training\Lib\site-packages\ultralytics\utils\loss.py:390: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


      55/60      1.59G      2.307      1.972      1.737         84        640: 100% ━━━━━━━━━━━━ 800/800 4.2it/s 3:09<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 7.1it/s 14.0s.1s
                   all        200      20826      0.198      0.101     0.0574     0.0221
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 40, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

55 epochs completed in 3.336 hours.
Optimizer stripped from D:\R26-IT-091\R26-IT-091\backend\runs\blueprint_model_gpu\weights\last.pt, 52.0MB
Optimizer stripped from D:\R26-IT-091\R26-IT-091\backend\runs\blueprint_model_gpu\weights\best.pt, 52.0MB

Validating D:\R26-IT-091\R26-IT-091\backend\runs\blueprint_model_gpu\weights\best.pt...
Ultralytics 8.4.48  Python-3.12.6 torch